# 📈 Time Series Forecasting & A/B Testing

**Learning Objectives:**
- Understand time series analysis fundamentals
- Implement moving averages and exponential smoothing
- Apply ARIMA and Prophet forecasting models
- Compare multiple forecasting methods
- Conduct A/B tests for manufacturing process improvements
- Interpret forecast confidence intervals

**Google Data Analytics Topics Covered:**
- Time series analysis and forecasting
- Trend and seasonality detection
- Statistical modeling (ARIMA)
- A/B testing and hypothesis testing
- Model selection and evaluation
- Confidence intervals and uncertainty quantification

---

## 1. Import Libraries

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# POSIVA modules
from src.statistical.time_series import TimeSeriesAnalyzer, ABTestAnalyzer
from src.statistical.forecasting import ARIMAForecaster, ExponentialSmoothingForecaster, ForecastComparator

# Try to import Prophet
try:
    from src.statistical.forecasting import ProphetForecaster
    PROPHET_AVAILABLE = True
except ImportError:
    PROPHET_AVAILABLE = False
    print("⚠️ Prophet not available. Install with: pip install prophet")

print("✅ Libraries imported successfully")
print(f"Prophet available: {PROPHET_AVAILABLE}")

## 2. Load and Prepare Data

We'll load semiconductor test data and aggregate by date to create time series.

In [ ]:
# Load data
df = pd.read_csv('../data/sample/sample_data.csv')

# Create synthetic timestamps (since sample data doesn't have dates)
# Simulate daily test data over 90 days
np.random.seed(42)
start_date = pd.Timestamp('2024-01-01')
df['timestamp'] = start_date + pd.to_timedelta(np.random.randint(0, 90, len(df)), unit='D')
df['date'] = df['timestamp'].dt.date

print(f"Data loaded: {len(df)} records")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nFirst few records:")
df[['lot_id', 'device_id', 'test_name', 'result', 'date']].head()

### Create Daily Yield Time Series

In [ ]:
# Calculate daily yield
daily_yield = df.groupby('date').agg({
    'result': lambda x: (x == 'PASS').mean() * 100,  # Yield percentage
    'device_id': 'nunique',  # Number of devices tested
    'test_time_ms': 'mean'  # Average test time
}).round(2)

daily_yield.columns = ['yield_pct', 'devices_tested', 'avg_test_time']

# Convert to time series with datetime index
daily_yield.index = pd.to_datetime(daily_yield.index)
daily_yield = daily_yield.sort_index()

print(f"Daily yield time series: {len(daily_yield)} days")
print(f"\nSummary statistics:")
print(daily_yield.describe().round(2))

# Plot
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Yield
axes[0].plot(daily_yield.index, daily_yield['yield_pct'], marker='o', linewidth=2, markersize=4)
axes[0].set_title('Daily Yield %', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Yield %')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=daily_yield['yield_pct'].mean(), color='r', linestyle='--', label='Mean')
axes[0].legend()

# Devices tested
axes[1].bar(daily_yield.index, daily_yield['devices_tested'], alpha=0.7, color='#3498db')
axes[1].set_title('Devices Tested Per Day', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].grid(True, alpha=0.3, axis='y')

# Test time
axes[2].plot(daily_yield.index, daily_yield['avg_test_time'], marker='s', linewidth=2, markersize=4, color='#e74c3c')
axes[2].set_title('Average Test Time (ms)', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Time (ms)')
axes[2].set_xlabel('Date')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\\n✅ Time series created successfully")

## 3. Exploratory Time Series Analysis

Check for trends, seasonality, and stationarity.

In [ ]:
# Initialize analyzer
ts_analyzer = TimeSeriesAnalyzer(df, date_col='timestamp')

# Trend analysis
trend_result = ts_analyzer.trend_analysis('result', period='D')

print("=" * 60)
print("TREND ANALYSIS")
print("=" * 60)
print(f"Slope: {trend_result['slope']:.6f}")
print(f"R-squared: {trend_result['r_squared']:.4f}")
print(f"P-value: {trend_result['p_value']:.4f}")
print(f"Trend: {trend_result['trend'].upper()}")
print(f"Change: {trend_result['change_pct']:.2f}%")

if trend_result['p_value'] < 0.05:
    direction = "improving" if trend_result['slope'] > 0 else "declining"
    print(f"\\n📊 Significant trend detected: Yield is {direction} over time")
else:
    print(f"\\n📊 No significant trend detected: Yield is stable")

# Seasonality detection
seasonality_result = ts_analyzer.seasonality_detection('result', period='D')

print("\\n" + "=" * 60)
print("SEASONALITY ANALYSIS")
print("=" * 60)
print(f"Has seasonality: {seasonality_result['has_seasonality']}")
print(f"Max autocorrelation: {seasonality_result['max_correlation']:.4f}")
print(f"Lag: {seasonality_result['lag_at_max']} days")

if seasonality_result['has_seasonality']:
    print(f"\\n📊 Weekly pattern detected with {seasonality_result['lag_at_max']}-day cycle")
else:
    print(f"\\n📊 No strong seasonality detected")

## 4. Simple Forecasting Methods

### 4.1 Moving Average

In [ ]:
# Calculate moving averages
ma_3 = ts_analyzer.moving_average('result', period='D', window=3)
ma_7 = ts_analyzer.moving_average('result', period='D', window=7)
ma_14 = ts_analyzer.moving_average('result', period='D', window=14)

# Aggregate yield by day for visualization
daily_pass_rate = df.groupby(df['timestamp'].dt.date)['result'].apply(lambda x: (x == 'PASS').mean()).reset_index()
daily_pass_rate.columns = ['date', 'yield']
daily_pass_rate['date'] = pd.to_datetime(daily_pass_rate['date'])

# Plot
plt.figure(figsize=(14, 6))
plt.plot(daily_pass_rate['date'], daily_pass_rate['yield'], 'o-', label='Actual Yield', alpha=0.5, markersize=4)
plt.plot(ma_3.index, ma_3.values, '-', linewidth=2, label='MA(3)')
plt.plot(ma_7.index, ma_7.values, '-', linewidth=2, label='MA(7)')
plt.plot(ma_14.index, ma_14.values, '-', linewidth=2, label='MA(14)')

plt.title('Moving Average Smoothing', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Yield Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Moving averages calculated")
print(f"MA(3) - Smooths out daily noise")
print(f"MA(7) - Weekly trend")
print(f"MA(14) - Bi-weekly trend")

### 4.2 Exponential Smoothing

In [ ]:
# Calculate exponential smoothing with different alphas
es_02 = ts_analyzer.exponential_smoothing('result', period='D', alpha=0.2)
es_05 = ts_analyzer.exponential_smoothing('result', period='D', alpha=0.5)
es_08 = ts_analyzer.exponential_smoothing('result', period='D', alpha=0.8)

# Plot
plt.figure(figsize=(14, 6))
plt.plot(daily_pass_rate['date'], daily_pass_rate['yield'], 'o-', label='Actual Yield', alpha=0.5, markersize=4)
plt.plot(es_02.index, es_02.values, '-', linewidth=2, label='EMA (α=0.2, more smoothing)')
plt.plot(es_05.index, es_05.values, '-', linewidth=2, label='EMA (α=0.5, balanced)')
plt.plot(es_08.index, es_08.values, '-', linewidth=2, label='EMA (α=0.8, less smoothing)')

plt.title('Exponential Moving Average with Different Alpha Values', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Yield Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Exponential smoothing calculated")
print(f"\\nAlpha interpretation:")
print(f"  • α = 0.2: More weight on historical data (smoother)")
print(f"  • α = 0.5: Balanced between recent and historical")
print(f"  • α = 0.8: More weight on recent data (responsive)")

### 4.3 Simple Forecasting

In [ ]:
# Simple forecast using different methods
forecast_results = ts_analyzer.forecast_simple('result', period='D', steps=14)

print("=" * 60)
print("14-DAY FORECAST")
print("=" * 60)

for method, values in forecast_results.items():
    print(f"\\n{method.upper()}:")
    print(f"  Mean forecast: {values.mean():.4f}")
    print(f"  Min: {values.min():.4f}, Max: {values.max():.4f}")
    print(f"  First 3 days: {values[:3].tolist()}")

# Visualize forecasts
fig = ts_analyzer.plot_forecast(forecast_results, 'result', period='D')
fig.show()

print("\\n✅ Simple forecasts generated")

## 5. ARIMA Forecasting

ARIMA (AutoRegressive Integrated Moving Average) is a powerful statistical model for time series forecasting.

In [ ]:
# Prepare yield time series
yield_ts = daily_yield['yield_pct']

# Split train/test (80/20)
train_size = int(len(yield_ts) * 0.8)
train = yield_ts[:train_size]
test = yield_ts[train_size:]

print(f"Train size: {len(train)} days")
print(f"Test size: {len(test)} days")

# Initialize ARIMA forecaster
arima = ARIMAForecaster()

# Check stationarity
stationarity = arima.check_stationarity(train)

print("\\n" + "=" * 60)
print("STATIONARITY TEST (Augmented Dickey-Fuller)")
print("=" * 60)
print(f"ADF Statistic: {stationarity['adf_statistic']:.4f}")
print(f"P-value: {stationarity['p_value']:.4f}")
print(f"Result: {stationarity['interpretation']}")
print(f"Is stationary: {stationarity['is_stationary']}")

if stationarity['is_stationary']:
    print("\\n✅ Series is stationary - ready for ARIMA modeling")
else:
    print("\\n⚠️ Series needs differencing to become stationary")

### Fit ARIMA Model

In [ ]:
# Fit ARIMA with auto-detected order
arima.fit(train, auto_order=True)

# Get model metrics
metrics = arima.get_metrics()

print("=" * 60)
print("ARIMA MODEL")
print("=" * 60)
print(f"Order (p, d, q): {metrics['order']}")
print(f"AIC: {metrics['aic']:.2f} (lower is better)")
print(f"BIC: {metrics['bic']:.2f} (lower is better)")

# Generate forecast
forecast_df = arima.predict(steps=len(test))

print(f"\\n✅ Forecast generated for {len(test)} days")
print(f"\\nForecast summary:")
print(f"  Mean: {forecast_df['forecast'].mean():.2f}%")
print(f"  95% CI width: {(forecast_df['upper_bound'] - forecast_df['lower_bound']).mean():.2f}%")

### Visualize ARIMA Forecast

In [ ]:
# Plot forecast
fig = arima.plot_forecast(steps=len(test))
fig.show()

# Calculate forecast accuracy on test set
mae = np.mean(np.abs(test.values - forecast_df['forecast'].values))
rmse = np.sqrt(np.mean((test.values - forecast_df['forecast'].values) ** 2))
mape = np.mean(np.abs((test.values - forecast_df['forecast'].values) / test.values)) * 100

print("\\n" + "=" * 60)
print("FORECAST ACCURACY (on test set)")
print("=" * 60)
print(f"MAE:  {mae:.2f}%")
print(f"RMSE: {rmse:.2f}%")
print(f"MAPE: {mape:.2f}%")

if mape < 5:
    print("\\n✅ Excellent forecast accuracy (MAPE < 5%)")
elif mape < 10:
    print("\\n✅ Good forecast accuracy (MAPE < 10%)")
else:
    print("\\n⚠️ Moderate forecast accuracy")

## 6. Exponential Smoothing Forecaster

In [ ]:
# Initialize Exponential Smoothing forecaster
es_forecaster = ExponentialSmoothingForecaster()

# Fit model with trend and seasonality
es_forecaster.fit(train, trend='add', seasonal='add', seasonal_periods=7)

# Generate forecast
es_forecast = es_forecaster.predict(steps=len(test))

# Plot
fig = es_forecaster.plot_forecast(steps=len(test))
fig.show()

# Calculate accuracy
es_mae = np.mean(np.abs(test.values - es_forecast.values))
es_rmse = np.sqrt(np.mean((test.values - es_forecast.values) ** 2))
es_mape = np.mean(np.abs((test.values - es_forecast.values) / test.values)) * 100

print("=" * 60)
print("EXPONENTIAL SMOOTHING FORECAST")
print("=" * 60)
print(f"MAE:  {es_mae:.2f}%")
print(f"RMSE: {es_rmse:.2f}%")
print(f"MAPE: {es_mape:.2f}%")

print(f"\\n✅ Exponential Smoothing forecast complete")

## 7. Compare Forecasting Methods

In [ ]:
# Create forecast comparator
comparator = ForecastComparator(train, test)

# Add forecasts
comparator.add_forecast('ARIMA', forecast_df['forecast'])
comparator.add_forecast('Exp. Smoothing', es_forecast)

# Calculate metrics
metrics_df = comparator.calculate_metrics()

print("=" * 60)
print("FORECAST METHOD COMPARISON")
print("=" * 60)
print(metrics_df.to_string(index=False))

# Plot comparison
fig = comparator.plot_comparison()
fig.show()

# Determine best method
best_method = metrics_df.iloc[0]['method']
best_rmse = metrics_df.iloc[0]['rmse']

print(f"\\n🏆 Best method: {best_method} (RMSE = {best_rmse:.2f}%)")
print(f"\\n✅ Model comparison complete")

## 8. A/B Testing for Process Improvement

Test if a new manufacturing process (Lot B) improves yield compared to the standard process (Lot A).

In [ ]:
# Initialize A/B test analyzer
ab_analyzer = ABTestAnalyzer()

# Get unique lots
lots = df['lot_id'].unique()
print(f"Available lots: {lots}")

# Select two lots for A/B comparison
lot_a_id = lots[0]
lot_b_id = lots[1] if len(lots) > 1 else lots[0]

# Calculate yield for each lot (device-level)
lot_a_yield = df[df['lot_id'] == lot_a_id].groupby('device_id')['result'].apply(lambda x: (x == 'PASS').mean())
lot_b_yield = df[df['lot_id'] == lot_b_id].groupby('device_id')['result'].apply(lambda x: (x == 'PASS').mean())

print(f"\\nLot A ({lot_a_id}): {len(lot_a_yield)} devices")
print(f"Lot B ({lot_b_id}): {len(lot_b_yield)} devices")

# Perform A/B test
ab_results = ab_analyzer.analyze_ab_test(lot_a_yield, lot_b_yield, metric_name='Device Yield')

print("\\n" + "=" * 60)
print("A/B TEST RESULTS")
print("=" * 60)
print(f"\\nGroup A (Control): Lot {lot_a_id}")
print(f"  Mean yield: {lot_a_yield.mean():.4f}")
print(f"  Std dev: {lot_a_yield.std():.4f}")
print(f"  Sample size: {ab_results['group_a_size']}")

print(f"\\nGroup B (Treatment): Lot {lot_b_id}")
print(f"  Mean yield: {lot_b_yield.mean():.4f}")
print(f"  Std dev: {lot_b_yield.std():.4f}")
print(f"  Sample size: {ab_results['group_b_size']}")

print(f"\\nStatistical Test:")
print(f"  Difference: {ab_results['difference']:.4f}")
print(f"  Improvement: {ab_results['improvement_pct']:.2f}%")
print(f"  T-statistic: {ab_results['t_statistic']:.4f}")
print(f"  P-value: {ab_results['p_value']:.4f}")
print(f"  Significant: {ab_results['significant']}")
print(f"  Cohen's d: {ab_results['cohens_d']:.4f} ({ab_results['effect_size_interpretation']})")

print(f"\\n{'='*60}")
print(f"RECOMMENDATION: {ab_results['recommendation']}")
print(f"{'='*60}")

if ab_results['significant'] and ab_results['improvement_pct'] > 0:
    print(f"\\n✅ Lot B shows statistically significant improvement!")
    print(f"   Expected yield gain: {ab_results['improvement_pct']:.2f}%")
elif ab_results['significant'] and ab_results['improvement_pct'] < 0:
    print(f"\\n⚠️ Lot B shows statistically significant decline!")
    print(f"   Expected yield loss: {abs(ab_results['improvement_pct']):.2f}%")
else:
    print(f"\\n📊 No statistically significant difference detected")
    print(f"   Need more data or effect is too small")

### Sample Size Calculation for Future A/B Tests

In [ ]:
# Calculate required sample size for different effect sizes
print("=" * 60)
print("SAMPLE SIZE RECOMMENDATIONS")
print("=" * 60)
print("\\nFor future A/B tests, required sample sizes:")
print(f"(Power = 0.80, Significance = 0.05)\\n")

for effect_size in [0.2, 0.5, 0.8]:
    sample_size_result = ab_analyzer.sample_size_calculator(
        effect_size=effect_size,
        alpha=0.05,
        power=0.80
    )
    
    effect_label = {0.2: 'Small', 0.5: 'Medium', 0.8: 'Large'}[effect_size]
    
    print(f"{effect_label} effect (d={effect_size}):")
    print(f"  Sample size per group: {sample_size_result['sample_size_per_group']}")
    print(f"  Total sample size: {sample_size_result['total_sample_size']}")
    print(f"  Minimum detectable difference: ~{effect_size * lot_a_yield.std():.4f} ({effect_size * lot_a_yield.std() / lot_a_yield.mean() * 100:.1f}%)\\n")

print("\\n✅ A/B testing analysis complete")

## 9. Key Takeaways

### Time Series Forecasting
1. **Stationarity**: Check before ARIMA modeling using ADF test
2. **Model Selection**: Compare multiple methods (ARIMA, Exponential Smoothing, Prophet)
3. **Validation**: Always validate on hold-out test set
4. **Confidence Intervals**: Report uncertainty in forecasts
5. **Seasonality**: Consider weekly/monthly patterns in manufacturing

### A/B Testing
1. **Statistical Significance**: p-value < 0.05 indicates real difference
2. **Effect Size**: Cohen's d quantifies practical significance
3. **Sample Size**: Calculate required sample size before testing
4. **Confidence**: Use confidence intervals, not just p-values
5. **Context**: Consider practical/business significance, not just statistical

### Semiconductor Applications
- **Yield Forecasting**: Predict future yield for capacity planning
- **Process Improvement**: A/B test new manufacturing processes
- **Anomaly Detection**: Forecast deviation from expected trend
- **Resource Optimization**: Forecast test time for scheduling
- **Quality Control**: Early warning system for yield degradation

---

**Next Steps:**
1. Try Prophet forecasting if available (better for strong seasonality)
2. Implement SARIMA for seasonal patterns
3. Create automated forecasting pipeline
4. Build dashboard page for forecast visualization
5. Set up alerting when actual deviates from forecast